# 🧭 Akbarxon AI Living Advisor — Uzbekistan

**Author:** Akbarxon Nasirov  
**Mentor:** Dr. Qingyang Xiao

This single Google Colab notebook builds a complete portfolio-ready prototype for an **AI-based living advisor platform**. Users describe their ideal lifestyle in natural language, and the system recommends cities in Uzbekistan, explains the ranking, visualizes results on a map, and learns from like/dislike feedback.

The generated Streamlit app displays the project title, author, and mentor in the left sidebar and repeats the team information in the **AI laboratory** tab.

> **Important:** The included city scores are illustrative, not official statistics. A public production app must replace them with licensed, dated, auditable data and show sources and update timestamps.


## 1. Install the required packages

Run this cell first in Google Colab. The notebook is designed for Python 3.12-compatible Colab runtimes.

In [ ]:
%pip install -q \
  "streamlit>=1.58,<2.0" \
  "pandas>=2.2,<3.0" \
  "numpy>=2.0,<3.0" \
  "scikit-learn>=1.6,<2.0" \
  "plotly>=6.0,<7.0" \
  "folium>=0.19,<1.0" \
  "streamlit-folium>=0.24,<1.0" \
  "requests>=2.32,<3.0" \
  "joblib>=1.4,<2.0"

In [ ]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())

## 2. Create the Uzbekistan city prototype dataset

The values below are intentionally labeled as **prototype scores from 0 to 100**. Higher is better for every dimension, including affordability. They are suitable for software demonstration, not relocation decisions.

In [ ]:
from pathlib import Path
import pandas as pd

rows = [
    {
        "city": "Tashkent", "region": "Tashkent City", "lat": 41.2995, "lon": 69.2401,
        "affordability": 52, "grocery_affordability": 56, "entertainment": 94, "sunset_scenery": 70,
        "safety": 76, "jobs": 96, "internet": 93, "healthcare": 95, "education": 96,
        "heritage": 78, "nature": 66, "climate_comfort": 64, "quietness": 38, "mobility": 96,
        "tags": "capital metro jobs universities hospitals restaurants nightlife shopping remote work international airport museums parks",
        "description": "Uzbekistan's largest urban center, with the strongest mix of jobs, universities, healthcare, public transport, dining, and entertainment, but a higher prototype cost profile and a busier pace."
    },
    {
        "city": "Samarkand", "region": "Samarkand Region", "lat": 39.6542, "lon": 66.9597,
        "affordability": 70, "grocery_affordability": 72, "entertainment": 82, "sunset_scenery": 91,
        "safety": 80, "jobs": 72, "internet": 76, "healthcare": 77, "education": 80,
        "heritage": 99, "nature": 74, "climate_comfort": 70, "quietness": 61, "mobility": 83,
        "tags": "silk road registan history architecture tourism sunset restaurants culture train airport walkable old city",
        "description": "A major Silk Road city combining world-famous architecture, strong tourism activity, scenic evenings, restaurants, universities, and good intercity connections."
    },
    {
        "city": "Bukhara", "region": "Bukhara Region", "lat": 39.7681, "lon": 64.4556,
        "affordability": 74, "grocery_affordability": 76, "entertainment": 72, "sunset_scenery": 92,
        "safety": 82, "jobs": 65, "internet": 72, "healthcare": 72, "education": 73,
        "heritage": 98, "nature": 58, "climate_comfort": 61, "quietness": 73, "mobility": 76,
        "tags": "historic center silk road old city sunset courtyards culture tourism calm affordable markets architecture",
        "description": "A compact historic city with a calm atmosphere, strong cultural identity, memorable sunsets over old architecture, and a relatively affordable prototype profile."
    },
    {
        "city": "Khiva", "region": "Khorezm Region", "lat": 41.3783, "lon": 60.3639,
        "affordability": 77, "grocery_affordability": 78, "entertainment": 63, "sunset_scenery": 97,
        "safety": 84, "jobs": 53, "internet": 66, "healthcare": 62, "education": 61,
        "heritage": 100, "nature": 56, "climate_comfort": 57, "quietness": 82, "mobility": 62,
        "tags": "itchan kala walled city heritage sunset photography quiet tourism traditional architecture walkable",
        "description": "A small walled heritage city especially strong for historic atmosphere, photography, sunset views, walkability, and a slower lifestyle."
    },
    {
        "city": "Nukus", "region": "Karakalpakstan", "lat": 42.4600, "lon": 59.6166,
        "affordability": 84, "grocery_affordability": 82, "entertainment": 52, "sunset_scenery": 80,
        "safety": 79, "jobs": 56, "internet": 65, "healthcare": 64, "education": 66,
        "heritage": 76, "nature": 68, "climate_comfort": 48, "quietness": 84, "mobility": 58,
        "tags": "savitsky museum karakalpak culture desert quiet affordable art remote regional center",
        "description": "A quieter and more affordable regional capital known for distinctive art and Karakalpak culture, with a remote desert setting and fewer big-city services."
    },
    {
        "city": "Fergana", "region": "Fergana Region", "lat": 40.3894, "lon": 71.7870,
        "affordability": 78, "grocery_affordability": 83, "entertainment": 69, "sunset_scenery": 78,
        "safety": 82, "jobs": 70, "internet": 74, "healthcare": 75, "education": 76,
        "heritage": 69, "nature": 86, "climate_comfort": 74, "quietness": 70, "mobility": 73,
        "tags": "fergana valley green parks markets family friendly affordable food regional services nature",
        "description": "A greener regional city in the Fergana Valley with strong markets, affordable groceries, family-oriented neighborhoods, and good access to valley destinations."
    },
    {
        "city": "Andijan", "region": "Andijan Region", "lat": 40.7821, "lon": 72.3442,
        "affordability": 80, "grocery_affordability": 85, "entertainment": 66, "sunset_scenery": 72,
        "safety": 80, "jobs": 72, "internet": 73, "healthcare": 74, "education": 75,
        "heritage": 71, "nature": 77, "climate_comfort": 71, "quietness": 66, "mobility": 72,
        "tags": "fergana valley commerce markets affordable groceries family business regional center parks",
        "description": "A commercially active valley city with strong local markets, a relatively affordable prototype cost profile, family services, and regional business opportunities."
    },
    {
        "city": "Namangan", "region": "Namangan Region", "lat": 40.9983, "lon": 71.6726,
        "affordability": 81, "grocery_affordability": 84, "entertainment": 67, "sunset_scenery": 76,
        "safety": 82, "jobs": 69, "internet": 72, "healthcare": 73, "education": 74,
        "heritage": 73, "nature": 83, "climate_comfort": 73, "quietness": 71, "mobility": 69,
        "tags": "gardens flowers valley nature affordable markets family calm regional city parks",
        "description": "A large but comparatively calm valley city associated with gardens, markets, family life, and access to greener landscapes."
    },
    {
        "city": "Qarshi", "region": "Qashqadaryo Region", "lat": 38.8606, "lon": 65.7891,
        "affordability": 83, "grocery_affordability": 82, "entertainment": 58, "sunset_scenery": 75,
        "safety": 80, "jobs": 67, "internet": 68, "healthcare": 69, "education": 69,
        "heritage": 68, "nature": 61, "climate_comfort": 55, "quietness": 78, "mobility": 68,
        "tags": "affordable quiet regional center local markets industry railway warm climate",
        "description": "An affordable regional center with a practical, quieter lifestyle, local industry, railway connections, and fewer entertainment options than the major tourism cities."
    },
    {
        "city": "Termez", "region": "Surxondaryo Region", "lat": 37.2242, "lon": 67.2783,
        "affordability": 82, "grocery_affordability": 80, "entertainment": 55, "sunset_scenery": 88,
        "safety": 78, "jobs": 61, "internet": 66, "healthcare": 67, "education": 67,
        "heritage": 84, "nature": 75, "climate_comfort": 46, "quietness": 79, "mobility": 61,
        "tags": "southern city archaeology buddhist heritage amu darya sunset warm climate quiet border region",
        "description": "A southern city with important archaeological heritage, dramatic river and desert light, warm weather, and a slower regional lifestyle."
    },
    {
        "city": "Jizzakh", "region": "Jizzakh Region", "lat": 40.1158, "lon": 67.8422,
        "affordability": 85, "grocery_affordability": 84, "entertainment": 54, "sunset_scenery": 79,
        "safety": 82, "jobs": 60, "internet": 67, "healthcare": 66, "education": 67,
        "heritage": 59, "nature": 88, "climate_comfort": 69, "quietness": 84, "mobility": 72,
        "tags": "mountains zaamin nature hiking affordable quiet highway family fresh air",
        "description": "A practical and affordable city with strong access to mountain and nature destinations, a quieter pace, and convenient east-west road and rail positioning."
    },
    {
        "city": "Gulistan", "region": "Sirdaryo Region", "lat": 40.4897, "lon": 68.7842,
        "affordability": 88, "grocery_affordability": 87, "entertainment": 47, "sunset_scenery": 68,
        "safety": 83, "jobs": 57, "internet": 66, "healthcare": 64, "education": 64,
        "heritage": 48, "nature": 59, "climate_comfort": 62, "quietness": 88, "mobility": 70,
        "tags": "very affordable quiet small city railway agriculture low cost calm",
        "description": "A smaller, quiet regional capital with one of the strongest illustrative affordability profiles, straightforward transport links, and limited nightlife."
    },
    {
        "city": "Navoi", "region": "Navoi Region", "lat": 40.0844, "lon": 65.3792,
        "affordability": 72, "grocery_affordability": 74, "entertainment": 61, "sunset_scenery": 79,
        "safety": 82, "jobs": 81, "internet": 75, "healthcare": 74, "education": 72,
        "heritage": 57, "nature": 60, "climate_comfort": 58, "quietness": 73, "mobility": 73,
        "tags": "industry mining jobs planned city airport railway parks practical career",
        "description": "A planned industrial city with comparatively strong employment potential, orderly urban form, parks, and useful air and rail links."
    },
    {
        "city": "Urgench", "region": "Khorezm Region", "lat": 41.5500, "lon": 60.6333,
        "affordability": 79, "grocery_affordability": 80, "entertainment": 62, "sunset_scenery": 79,
        "safety": 81, "jobs": 64, "internet": 70, "healthcare": 70, "education": 70,
        "heritage": 76, "nature": 57, "climate_comfort": 56, "quietness": 76, "mobility": 76,
        "tags": "gateway to khiva airport railway affordable markets regional services khorezm",
        "description": "A practical service and transport base for Khorezm, offering airport and rail access, local markets, and convenient proximity to Khiva."
    },
    {
        "city": "Kokand", "region": "Fergana Region", "lat": 40.5286, "lon": 70.9425,
        "affordability": 82, "grocery_affordability": 84, "entertainment": 64, "sunset_scenery": 75,
        "safety": 81, "jobs": 66, "internet": 70, "healthcare": 69, "education": 71,
        "heritage": 88, "nature": 72, "climate_comfort": 71, "quietness": 72, "mobility": 74,
        "tags": "khanate palace heritage valley markets affordable traditional crafts railway",
        "description": "A historic Fergana Valley city with palace architecture, traditional crafts, markets, and a balanced blend of affordability and cultural interest."
    },
    {
        "city": "Shahrisabz", "region": "Qashqadaryo Region", "lat": 39.0578, "lon": 66.8342,
        "affordability": 84, "grocery_affordability": 83, "entertainment": 57, "sunset_scenery": 89,
        "safety": 83, "jobs": 56, "internet": 65, "healthcare": 63, "education": 64,
        "heritage": 93, "nature": 90, "climate_comfort": 72, "quietness": 86, "mobility": 63,
        "tags": "amir timur heritage mountains scenic sunset quiet affordable gardens history",
        "description": "A smaller historic city south of Samarkand with mountain scenery, major Timurid heritage, attractive sunsets, and a quiet lifestyle."
    },
]

output = Path("cities_uzbekistan.csv")
pd.DataFrame(rows).to_csv(output, index=False)
print(f"Wrote {output} with {len(rows)} cities")


In [ ]:
import pandas as pd

cities = pd.read_csv('cities_uzbekistan.csv')
print(f'Loaded {len(cities)} city profiles and {len(cities.columns)} columns.')
display(cities.head(8))

## 3. Generate the complete Streamlit application

The app contains four main tabs:

- **Recommend:** natural-language ranking, explanations, and feedback
- **Map & compare:** OpenStreetMap/Folium visualization and radar charts
- **City explorer:** detailed score profiles for all cities
- **AI laboratory:** machine-learning clusters and model explanation

No paid API key is required. Optional live-data calls fail safely when unavailable.

In [ ]:
%%writefile app.py
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
from urllib.parse import quote

import folium
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import streamlit as st
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from streamlit_folium import st_folium

APP_DIR = Path(__file__).resolve().parent
DATA_PATH = APP_DIR / "cities_uzbekistan.csv"
FEEDBACK_PATH = APP_DIR / "feedback_state.json"
MODEL_PATH = APP_DIR / "demo_preference_dnn.joblib"

FEATURE_COLUMNS = [
    "affordability",
    "grocery_affordability",
    "entertainment",
    "sunset_scenery",
    "safety",
    "jobs",
    "internet",
    "healthcare",
    "education",
    "heritage",
    "nature",
    "climate_comfort",
    "quietness",
    "mobility",
]

FEATURE_LABELS = {
    "affordability": "Affordable living",
    "grocery_affordability": "Affordable groceries",
    "entertainment": "Entertainment",
    "sunset_scenery": "Sunset & scenery",
    "safety": "Safety",
    "jobs": "Jobs & career",
    "internet": "Internet & remote work",
    "healthcare": "Healthcare",
    "education": "Education",
    "heritage": "History & culture",
    "nature": "Nature access",
    "climate_comfort": "Climate comfort",
    "quietness": "Quiet lifestyle",
    "mobility": "Transportation",
}

SYNONYMS = {
    "affordability": [
        "cheap", "cheapest", "affordable", "low cost", "low-cost", "budget",
        "inexpensive", "save money", "low rent", "reasonable rent",
    ],
    "grocery_affordability": [
        "cheap grocery", "cheap groceries", "grocery price", "food price",
        "affordable food", "low food cost", "market price", "cheap market",
    ],
    "entertainment": [
        "entertainment", "nightlife", "fun", "activities", "events", "concert",
        "cinema", "shopping", "restaurants", "social life", "things to do",
    ],
    "sunset_scenery": [
        "sunset", "sunsets", "scenic", "beautiful view", "view", "photography",
        "sky", "romantic", "landscape",
    ],
    "safety": ["safe", "safety", "secure", "low crime", "family friendly"],
    "jobs": ["job", "jobs", "career", "employment", "business", "salary", "startup"],
    "internet": [
        "internet", "wifi", "remote work", "digital nomad", "online work",
        "technology", "tech", "fast connection",
    ],
    "healthcare": ["hospital", "healthcare", "doctor", "medical", "clinic", "health"],
    "education": [
        "school", "education", "university", "college", "student", "children",
        "academic", "learning",
    ],
    "heritage": [
        "history", "historic", "heritage", "culture", "architecture", "museum",
        "old city", "silk road", "traditional",
    ],
    "nature": [
        "nature", "mountain", "mountains", "green", "park", "hiking", "outdoor",
        "river", "lake", "desert", "fresh air",
    ],
    "climate_comfort": [
        "comfortable weather", "mild climate", "climate", "weather", "not too hot",
        "not too cold", "pleasant weather",
    ],
    "quietness": ["quiet", "calm", "peaceful", "slow life", "less crowded", "relaxed"],
    "mobility": [
        "transport", "transportation", "metro", "bus", "airport", "walkable",
        "commute", "easy travel", "connected",
    ],
}

CLUSTER_NAMES = {
    0: "Balanced regional center",
    1: "Culture and tourism hub",
    2: "Affordable quiet city",
    3: "Career and services hub",
}


def inject_css() -> None:
    st.markdown(
        """
        <style>
        .block-container {padding-top: 1.4rem; padding-bottom: 3rem; max-width: 1250px;}
        .hero {
            padding: 1.35rem 1.5rem; border-radius: 22px;
            background: linear-gradient(135deg, #0f766e 0%, #0369a1 55%, #4338ca 100%);
            color: white; margin-bottom: 1rem;
            box-shadow: 0 12px 32px rgba(3, 105, 161, 0.18);
        }
        .hero h1 {margin: 0 0 .25rem 0; font-size: 2.2rem;}
        .hero p {margin: 0; opacity: .94; font-size: 1.02rem;}
        .city-card {
            border: 1px solid rgba(100,116,139,.22); border-radius: 18px;
            padding: 1rem 1.05rem; margin: .35rem 0 .85rem 0;
            background: rgba(255,255,255,.02);
        }
        .score-pill {
            display: inline-block; padding: .22rem .62rem; border-radius: 999px;
            background: rgba(14,165,233,.12); font-weight: 700;
        }
        .small-note {font-size: .86rem; opacity: .78;}
        .sidebar-brand {
            padding: .85rem .8rem .78rem .8rem;
            border: 1px solid rgba(14, 116, 144, .22);
            border-radius: 16px;
            background: linear-gradient(145deg, rgba(15,118,110,.10), rgba(67,56,202,.08));
            margin: .2rem 0 .65rem 0;
        }
        .sidebar-brand-title {
            font-size: 1.08rem; font-weight: 800; line-height: 1.25;
            margin-bottom: .55rem;
        }
        .sidebar-team {font-size: .92rem; line-height: 1.55;}
        .project-team-card {
            border: 1px solid rgba(67,56,202,.20);
            border-radius: 18px; padding: 1rem 1.15rem; margin: .2rem 0 1rem 0;
            background: linear-gradient(135deg, rgba(15,118,110,.08), rgba(3,105,161,.08), rgba(67,56,202,.08));
        }
        .project-team-card h3 {margin: 0 0 .55rem 0;}
        .project-team-card p {margin: 0; line-height: 1.65;}
        </style>
        """,
        unsafe_allow_html=True,
    )


@st.cache_data
def load_city_data() -> pd.DataFrame:
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "cities_uzbekistan.csv is missing. Run the Colab notebook export cells first."
        )
    df = pd.read_csv(DATA_PATH)
    missing = [column for column in FEATURE_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")
    for column in FEATURE_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(50).clip(0, 100)
    return df


def city_document(row: pd.Series) -> str:
    feature_words = " ".join(
        FEATURE_LABELS[column]
        for column in FEATURE_COLUMNS
        if float(row[column]) >= 72
    )
    return " ".join(
        [
            str(row.get("city", "")),
            str(row.get("region", "")),
            str(row.get("tags", "")),
            str(row.get("description", "")),
            feature_words,
        ]
    )


@st.cache_resource
def build_text_index(documents: Tuple[str, ...]) -> Tuple[TfidfVectorizer, object]:
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=1)
    matrix = vectorizer.fit_transform(documents)
    return vectorizer, matrix


def parse_preference_weights(query: str) -> Tuple[Dict[str, float], List[str]]:
    text = re.sub(r"\s+", " ", query.lower()).strip()
    weights = {feature: 0.0 for feature in FEATURE_COLUMNS}
    matches: List[str] = []

    for feature, phrases in SYNONYMS.items():
        for phrase in phrases:
            if phrase in text:
                increment = 1.4 if " " in phrase else 1.0
                weights[feature] += increment
                matches.append(f"{FEATURE_LABELS[feature]} ← '{phrase}'")

    # Small intent rules for common compound requests.
    if "family" in text or "children" in text or "kids" in text:
        weights["safety"] += 1.1
        weights["education"] += 0.9
        weights["healthcare"] += 0.6
        matches.append("Family intent → safety, education, healthcare")
    if "retire" in text or "retirement" in text:
        weights["affordability"] += 0.8
        weights["quietness"] += 1.0
        weights["healthcare"] += 0.8
        matches.append("Retirement intent → affordability, quietness, healthcare")
    if "young professional" in text or "professional" in text:
        weights["jobs"] += 1.0
        weights["internet"] += 0.7
        weights["entertainment"] += 0.5
        matches.append("Professional intent → jobs, internet, entertainment")
    if "tourist" in text or "tourism" in text or "visit" in text:
        weights["heritage"] += 0.9
        weights["entertainment"] += 0.5
        weights["mobility"] += 0.4
        matches.append("Tourism intent → heritage, activities, transportation")

    total = sum(weights.values())
    if total <= 0:
        defaults = {
            "affordability": 1.0,
            "safety": 1.0,
            "healthcare": 0.8,
            "internet": 0.7,
            "mobility": 0.6,
            "entertainment": 0.5,
        }
        weights.update(defaults)
        matches.append("No strong keyword found → balanced default profile")
        total = sum(weights.values())

    return {key: value / total for key, value in weights.items()}, matches


def combine_weights(
    text_weights: Dict[str, float], slider_weights: Dict[str, float]
) -> Dict[str, float]:
    combined = {}
    for feature in FEATURE_COLUMNS:
        # Text remains the primary signal; sliders act as explicit corrections.
        combined[feature] = text_weights.get(feature, 0.0) + slider_weights.get(feature, 0.0) / 5.0
    total = sum(combined.values()) or 1.0
    return {feature: value / total for feature, value in combined.items()}


def load_feedback_state(cities: Iterable[str]) -> Dict[str, Dict[str, int]]:
    default = {str(city): {"likes": 0, "dislikes": 0} for city in cities}
    if not FEEDBACK_PATH.exists():
        return default
    try:
        stored = json.loads(FEEDBACK_PATH.read_text(encoding="utf-8"))
        for city in default:
            values = stored.get(city, {})
            default[city]["likes"] = int(values.get("likes", 0))
            default[city]["dislikes"] = int(values.get("dislikes", 0))
    except (OSError, ValueError, TypeError):
        pass
    return default


def save_feedback_state(state: Dict[str, Dict[str, int]]) -> None:
    try:
        FEEDBACK_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
    except OSError:
        # Some hosted deployments have read-only or ephemeral filesystems.
        pass


def bandit_posterior(city: str, state: Dict[str, Dict[str, int]]) -> float:
    values = state.get(city, {"likes": 0, "dislikes": 0})
    likes = max(0, int(values.get("likes", 0)))
    dislikes = max(0, int(values.get("dislikes", 0)))
    return (likes + 1.0) / (likes + dislikes + 2.0)


def make_dnn_training_data(df: pd.DataFrame, n_samples: int = 2400) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(42)
    features = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    x_rows: List[np.ndarray] = []
    y_rows: List[int] = []

    for _ in range(n_samples):
        city_vector = features[rng.integers(0, len(features))]
        preference = rng.dirichlet(np.ones(len(FEATURE_COLUMNS)) * 0.75)
        interaction = preference * city_vector
        utility = float(np.dot(preference, city_vector))

        # Nonlinear lifestyle interactions create a useful neural-network demonstration.
        heritage_idx = FEATURE_COLUMNS.index("heritage")
        entertainment_idx = FEATURE_COLUMNS.index("entertainment")
        affordability_idx = FEATURE_COLUMNS.index("affordability")
        grocery_idx = FEATURE_COLUMNS.index("grocery_affordability")
        jobs_idx = FEATURE_COLUMNS.index("jobs")
        internet_idx = FEATURE_COLUMNS.index("internet")
        nature_idx = FEATURE_COLUMNS.index("nature")
        sunset_idx = FEATURE_COLUMNS.index("sunset_scenery")

        utility += 0.09 * min(
            preference[heritage_idx] * city_vector[heritage_idx],
            preference[entertainment_idx] * city_vector[entertainment_idx],
        )
        utility += 0.08 * min(
            preference[affordability_idx] * city_vector[affordability_idx],
            preference[grocery_idx] * city_vector[grocery_idx],
        )
        utility += 0.07 * min(
            preference[jobs_idx] * city_vector[jobs_idx],
            preference[internet_idx] * city_vector[internet_idx],
        )
        utility += 0.06 * min(
            preference[nature_idx] * city_vector[nature_idx],
            preference[sunset_idx] * city_vector[sunset_idx],
        )
        utility += rng.normal(0, 0.025)

        x_rows.append(np.concatenate([preference, city_vector, interaction]))
        like_probability = 1.0 / (1.0 + math.exp(-(utility - 0.73) / 0.055))
        y_rows.append(int(rng.random() < like_probability))

    return np.vstack(x_rows), np.asarray(y_rows, dtype=int)


@st.cache_resource
def train_or_load_demo_dnn(data_signature: str, _df: pd.DataFrame) -> Pipeline:
    del data_signature
    if MODEL_PATH.exists():
        try:
            return joblib.load(MODEL_PATH)
        except Exception:
            pass

    x_train, y_train = make_dnn_training_data(_df)
    model = Pipeline(
        steps=[
            ("scale", StandardScaler()),
            (
                "dnn",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32, 16),
                    activation="relu",
                    alpha=0.001,
                    learning_rate_init=0.002,
                    max_iter=350,
                    early_stopping=True,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(x_train, y_train)
    try:
        joblib.dump(model, MODEL_PATH)
    except OSError:
        pass
    return model


def dnn_like_probabilities(
    model: Pipeline, df: pd.DataFrame, weights: Dict[str, float]
) -> np.ndarray:
    preference = np.array([weights[feature] for feature in FEATURE_COLUMNS], dtype=float)
    city_features = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    x = np.hstack(
        [
            np.repeat(preference.reshape(1, -1), len(df), axis=0),
            city_features,
            city_features * preference,
        ]
    )
    return model.predict_proba(x)[:, 1]


def rank_cities(
    df: pd.DataFrame,
    query: str,
    slider_weights: Dict[str, float],
    feedback_state: Dict[str, Dict[str, int]],
    top_n: int,
) -> Tuple[pd.DataFrame, Dict[str, float], List[str]]:
    text_weights, matches = parse_preference_weights(query)
    weights = combine_weights(text_weights, slider_weights)

    documents = tuple(city_document(row) for _, row in df.iterrows())
    vectorizer, city_matrix = build_text_index(documents)
    query_vector = vectorizer.transform([query or "balanced affordable safe city"])
    text_similarity = cosine_similarity(query_vector, city_matrix).ravel()

    feature_matrix = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    weight_vector = np.array([weights[column] for column in FEATURE_COLUMNS], dtype=float)
    preference_score = feature_matrix @ weight_vector

    signature = str(hash(tuple(np.round(feature_matrix.ravel(), 4))))
    dnn_model = train_or_load_demo_dnn(signature, df)
    dnn_score = dnn_like_probabilities(dnn_model, df, weights)

    posterior = np.array(
        [bandit_posterior(city, feedback_state) for city in df["city"]], dtype=float
    )

    # Transparent hybrid score: explicit preferences dominate.
    final_score = (
        0.68 * preference_score
        + 0.14 * text_similarity
        + 0.13 * dnn_score
        + 0.05 * posterior
    )

    ranked = df.copy()
    ranked["preference_score"] = preference_score * 100
    ranked["text_similarity"] = text_similarity * 100
    ranked["dnn_like_probability"] = dnn_score * 100
    ranked["feedback_posterior"] = posterior * 100
    ranked["match_score"] = final_score * 100
    ranked = ranked.sort_values("match_score", ascending=False).head(top_n).reset_index(drop=True)
    return ranked, weights, matches


def top_reasons(row: pd.Series, weights: Dict[str, float], n: int = 4) -> List[str]:
    contributions = []
    for feature in FEATURE_COLUMNS:
        contribution = weights.get(feature, 0.0) * float(row[feature])
        contributions.append((contribution, feature, float(row[feature])))
    contributions.sort(reverse=True)
    reasons = []
    for _, feature, value in contributions[:n]:
        reasons.append(f"{FEATURE_LABELS[feature]}: {value:.0f}/100")
    return reasons


def build_map(df: pd.DataFrame, ranked: pd.DataFrame | None = None) -> folium.Map:
    map_object = folium.Map(
        location=[41.2, 64.6],
        zoom_start=5,
        tiles="OpenStreetMap",
        control_scale=True,
    )
    ranked_lookup = {}
    if ranked is not None:
        ranked_lookup = {
            city: (index + 1, float(score))
            for index, (city, score) in enumerate(zip(ranked["city"], ranked["match_score"]))
        }

    for _, row in df.iterrows():
        city = str(row["city"])
        rank_text = ""
        icon_color = "blue"
        if city in ranked_lookup:
            rank, score = ranked_lookup[city]
            rank_text = f"<br><b>Recommendation rank:</b> #{rank}<br><b>Match:</b> {score:.1f}/100"
            icon_color = "green" if rank == 1 else "cadetblue"
        popup = folium.Popup(
            f"<b>{city}</b><br>{row['region']}<br>{row['description']}{rank_text}",
            max_width=330,
        )
        folium.Marker(
            location=[float(row["lat"]), float(row["lon"])],
            tooltip=city,
            popup=popup,
            icon=folium.Icon(color=icon_color, icon="home"),
        ).add_to(map_object)
    return map_object


@st.cache_data(ttl=1800, show_spinner=False)
def fetch_live_weather(lat: float, lon: float) -> Dict[str, object]:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,apparent_temperature,weather_code,wind_speed_10m",
        "daily": "temperature_2m_max,temperature_2m_min,sunset",
        "forecast_days": 3,
        "timezone": "auto",
    }
    response = requests.get(url, params=params, timeout=7)
    response.raise_for_status()
    return response.json()


@st.cache_data(ttl=86400, show_spinner=False)
def fetch_wikipedia_summary(city: str) -> Dict[str, object]:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(city)}"
    response = requests.get(url, timeout=7, headers={"User-Agent": "LivingAdvisorPrototype/1.0"})
    response.raise_for_status()
    return response.json()


def weather_code_label(code: int | float | None) -> str:
    if code is None:
        return "Unknown"
    code = int(code)
    if code == 0:
        return "Clear sky"
    if code in {1, 2, 3}:
        return "Partly cloudy"
    if code in {45, 48}:
        return "Fog"
    if 51 <= code <= 67:
        return "Rain or drizzle"
    if 71 <= code <= 77:
        return "Snow"
    if 80 <= code <= 82:
        return "Rain showers"
    if code >= 95:
        return "Thunderstorm"
    return "Mixed conditions"


def radar_figure(row: pd.Series, features: List[str]) -> go.Figure:
    values = [float(row[feature]) for feature in features]
    labels = [FEATURE_LABELS[feature] for feature in features]
    values_closed = values + [values[0]]
    labels_closed = labels + [labels[0]]
    figure = go.Figure(
        data=[
            go.Scatterpolar(
                r=values_closed,
                theta=labels_closed,
                fill="toself",
                name=str(row["city"]),
            )
        ]
    )
    figure.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        showlegend=False,
        margin=dict(l=35, r=35, t=35, b=35),
        height=430,
    )
    return figure


def cluster_cities(df: pd.DataFrame) -> pd.DataFrame:
    model = KMeans(n_clusters=4, random_state=42, n_init=20)
    output = df.copy()
    output["cluster_id"] = model.fit_predict(df[FEATURE_COLUMNS])

    # Assign human-readable labels by cluster characteristics rather than numeric ID.
    summaries = output.groupby("cluster_id")[FEATURE_COLUMNS].mean()
    cluster_labels: Dict[int, str] = {}
    for cluster_id, values in summaries.iterrows():
        if values["jobs"] + values["healthcare"] + values["education"] >= 230:
            label = "Career and services hub"
        elif values["heritage"] + values["entertainment"] >= 145:
            label = "Culture and tourism hub"
        elif values["affordability"] + values["quietness"] >= 155:
            label = "Affordable quiet city"
        else:
            label = "Balanced regional center"
        cluster_labels[int(cluster_id)] = label
    output["city_archetype"] = output["cluster_id"].map(cluster_labels)
    return output


def render_feedback_buttons(city: str, feedback_state: Dict[str, Dict[str, int]], key_prefix: str) -> None:
    left, right, stats = st.columns([1, 1, 2])
    if left.button("👍 Like", key=f"{key_prefix}_like_{city}", use_container_width=True):
        feedback_state[city]["likes"] += 1
        save_feedback_state(feedback_state)
        st.toast(f"Feedback saved for {city}")
        st.rerun()
    if right.button("👎 Not for me", key=f"{key_prefix}_dislike_{city}", use_container_width=True):
        feedback_state[city]["dislikes"] += 1
        save_feedback_state(feedback_state)
        st.toast(f"Feedback saved for {city}")
        st.rerun()
    values = feedback_state[city]
    stats.caption(
        f"Prototype feedback: {values['likes']} likes · {values['dislikes']} dislikes · "
        f"bandit confidence {bandit_posterior(city, feedback_state):.2f}"
    )


def main() -> None:
    st.set_page_config(
        page_title="Akbarxon AI Living Advisor",
        page_icon="🧭",
        layout="wide",
        initial_sidebar_state="expanded",
    )
    inject_css()

    df = load_city_data()
    feedback_state = load_feedback_state(df["city"])

    st.markdown(
        """
        <div class="hero">
          <h1>🧭 Akbarxon AI Living Advisor</h1>
          <p>Describe the lifestyle you want, compare Uzbekistan cities, explore the map, and improve recommendations through feedback.</p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    with st.sidebar:
        st.markdown(
            """
            <div class="sidebar-brand">
              <div class="sidebar-brand-title">🧭 Akbarxon AI Living Advisor</div>
              <div class="sidebar-team">
                <strong>Author:</strong> Akbarxon Nasirov<br>
                <strong>Mentor:</strong> Dr. Qingyang Xiao
              </div>
            </div>
            """,
            unsafe_allow_html=True,
        )
        st.header("Your priorities")
        st.caption("Text is the main input. Sliders let you emphasize or correct specific priorities.")
        top_n = st.slider("Number of recommendations", 3, 8, 5)
        with st.expander("Advanced preference sliders", expanded=False):
            slider_weights = {
                feature: float(st.slider(FEATURE_LABELS[feature], 0, 5, 0, key=f"slider_{feature}"))
                for feature in FEATURE_COLUMNS
            }
        st.divider()
        st.info(
            "Prototype note: city scores are illustrative, not official statistics. "
            "Verify housing, employment, safety, healthcare, visa, and legal information before relocating."
        )

    recommend_tab, map_tab, explorer_tab, ai_tab = st.tabs(
        ["✨ Recommend", "🗺️ Map & compare", "🏙️ City explorer", "🧠 AI laboratory"]
    )

    with recommend_tab:
        query = st.text_area(
            "What kind of place are you looking for?",
            value="I want an affordable city with cheap groceries, beautiful sunsets, and good entertainment.",
            height=105,
            help="Examples: family-friendly and safe; best for remote work; historic and walkable; quiet retirement city.",
        )
        run = st.button("Find my best cities", type="primary", use_container_width=True)

        if run or "ranked_results" not in st.session_state:
            ranked, weights, matches = rank_cities(
                df, query, slider_weights, feedback_state, top_n
            )
            st.session_state["ranked_results"] = ranked
            st.session_state["active_weights"] = weights
            st.session_state["query_matches"] = matches
            st.session_state["active_query"] = query

        ranked = st.session_state["ranked_results"]
        weights = st.session_state["active_weights"]
        matches = st.session_state["query_matches"]

        with st.expander("How the AI interpreted your request", expanded=False):
            st.write("Detected signals:")
            for match in matches:
                st.write(f"- {match}")
            weight_table = pd.DataFrame(
                {
                    "Preference": [FEATURE_LABELS[key] for key in FEATURE_COLUMNS],
                    "Weight (%)": [round(weights[key] * 100, 1) for key in FEATURE_COLUMNS],
                }
            ).sort_values("Weight (%)", ascending=False)
            st.dataframe(weight_table, hide_index=True, use_container_width=True)

        st.subheader("Top matches")
        for index, row in ranked.iterrows():
            reasons = top_reasons(row, weights)
            st.markdown(
                f"""
                <div class="city-card">
                  <h3>#{index + 1} {row['city']} <span class="score-pill">{row['match_score']:.1f}/100 match</span></h3>
                  <p><b>{row['region']}</b> · {row['description']}</p>
                  <p><b>Why it fits:</b> {' · '.join(reasons)}</p>
                  <p class="small-note">Hybrid components: preference {row['preference_score']:.1f}, text {row['text_similarity']:.1f}, neural model {row['dnn_like_probability']:.1f}, feedback {row['feedback_posterior']:.1f}</p>
                </div>
                """,
                unsafe_allow_html=True,
            )
            render_feedback_buttons(str(row["city"]), feedback_state, f"rec_{index}")

        selected_live_city = st.selectbox(
            "Optional live context",
            ranked["city"].tolist(),
            help="Uses public Wikipedia and Open-Meteo endpoints. Data may be unavailable or incomplete.",
        )
        if st.button("Fetch live context for selected city"):
            city_row = df.loc[df["city"] == selected_live_city].iloc[0]
            with st.spinner("Retrieving live context..."):
                try:
                    wiki = fetch_wikipedia_summary(selected_live_city)
                    weather = fetch_live_weather(float(city_row["lat"]), float(city_row["lon"]))
                    current = weather.get("current", {})
                    col1, col2, col3 = st.columns(3)
                    col1.metric("Temperature", f"{current.get('temperature_2m', '—')} °C")
                    col2.metric("Feels like", f"{current.get('apparent_temperature', '—')} °C")
                    col3.metric("Conditions", weather_code_label(current.get("weather_code")))
                    st.write(wiki.get("extract", "No city summary was returned."))
                    if wiki.get("content_urls", {}).get("desktop", {}).get("page"):
                        st.link_button(
                            "Open source article",
                            wiki["content_urls"]["desktop"]["page"],
                        )
                except requests.RequestException as exc:
                    st.warning(f"Live data is temporarily unavailable: {exc}")

    with map_tab:
        ranked = st.session_state.get("ranked_results")
        st.subheader("Uzbekistan city map")
        st_folium(build_map(df, ranked), width=None, height=560, returned_objects=[])

        if ranked is not None:
            st.subheader("Recommendation score comparison")
            chart_df = ranked[["city", "match_score"]].sort_values("match_score")
            figure = px.bar(
                chart_df,
                x="match_score",
                y="city",
                orientation="h",
                labels={"match_score": "Match score", "city": "City"},
                range_x=[0, 100],
            )
            figure.update_layout(height=390, margin=dict(l=20, r=20, t=20, b=20))
            st.plotly_chart(figure, use_container_width=True)

            compare_names = st.multiselect(
                "Choose up to three cities for detailed comparison",
                df["city"].tolist(),
                default=ranked["city"].head(2).tolist(),
                max_selections=3,
            )
            comparison_features = [
                "affordability",
                "grocery_affordability",
                "entertainment",
                "sunset_scenery",
                "safety",
                "jobs",
                "internet",
                "healthcare",
                "heritage",
                "nature",
            ]
            columns = st.columns(max(1, len(compare_names)))
            for column, city in zip(columns, compare_names):
                row = df.loc[df["city"] == city].iloc[0]
                with column:
                    st.plotly_chart(radar_figure(row, comparison_features), use_container_width=True)
                    st.caption(row["description"])

    with explorer_tab:
        st.subheader("Explore every prototype city profile")
        city = st.selectbox("Select a city", df["city"].tolist(), key="city_explorer")
        row = df.loc[df["city"] == city].iloc[0]
        left, right = st.columns([1.1, 1])
        with left:
            st.markdown(f"### {row['city']}")
            st.write(f"**Region:** {row['region']}")
            st.write(row["description"])
            st.write(f"**Tags:** {row['tags']}")
            score_table = pd.DataFrame(
                {
                    "Dimension": [FEATURE_LABELS[feature] for feature in FEATURE_COLUMNS],
                    "Prototype score": [float(row[feature]) for feature in FEATURE_COLUMNS],
                }
            ).sort_values("Prototype score", ascending=False)
            st.dataframe(score_table, hide_index=True, use_container_width=True)
        with right:
            st.plotly_chart(
                radar_figure(
                    row,
                    [
                        "affordability",
                        "entertainment",
                        "sunset_scenery",
                        "safety",
                        "jobs",
                        "internet",
                        "healthcare",
                        "heritage",
                        "nature",
                    ],
                ),
                use_container_width=True,
            )
        render_feedback_buttons(city, feedback_state, "explorer")

    with ai_tab:
        st.markdown(
            """
            <div class="project-team-card">
              <h3>👥 Project team</h3>
              <p>
                <strong>Author:</strong> Akbarxon Nasirov<br>
                <strong>Mentor:</strong> Dr. Qingyang Xiao
              </p>
            </div>
            """,
            unsafe_allow_html=True,
        )
        st.subheader("What the prototype AI is doing")
        st.markdown(
            """
            **1. Natural-language preference parsing:** keyword and phrase rules convert a request into transparent lifestyle weights.

            **2. Text similarity:** TF–IDF and cosine similarity compare the user's request with each city profile.

            **3. Machine learning:** K-means groups cities into data-driven lifestyle archetypes.

            **4. Deep neural network demonstration:** a three-hidden-layer MLP predicts a like probability for each user–city pair. It is trained on synthetic prototype interactions until real, consented feedback is available.

            **5. Reinforcement-learning-style feedback:** a Beta-Bernoulli bandit updates each city's recommendation bonus after likes and dislikes.
            """
        )
        clustered = cluster_cities(df)
        st.dataframe(
            clustered[["city", "region", "city_archetype"] + FEATURE_COLUMNS],
            hide_index=True,
            use_container_width=True,
        )
        st.warning(
            "For production, replace illustrative scores and synthetic training data with licensed, dated, auditable sources. "
            "Use a real database for feedback, user accounts, privacy controls, source citations, and model monitoring."
        )

    st.divider()
    st.caption(
        "Author: Akbarxon Nasirov · Mentor: Dr. Qingyang Xiao · "
        "Built as an educational prototype for Akbarxon's GitHub portfolio. "
        "The app does not provide legal, immigration, housing, medical, or financial advice."
    )


if __name__ == "__main__":
    main()


## 4. Create GitHub and deployment files

In [ ]:
%%writefile requirements.txt
streamlit>=1.58,<2.0
pandas>=2.2,<3.0
numpy>=2.0,<3.0
scikit-learn>=1.6,<2.0
plotly>=6.0,<7.0
folium>=0.19,<1.0
streamlit-folium>=0.24,<1.0
requests>=2.32,<3.0
joblib>=1.4,<2.0


In [ ]:
%%writefile README.md
# Akbarxon AI Living Advisor for Uzbekistan

A Streamlit portfolio project that recommends cities in Uzbekistan from a user's natural-language living preferences.

## Project team

- **Author:** Akbarxon Nasirov
- **Mentor:** Dr. Qingyang Xiao

The author and mentor are displayed in two places in the web app:

1. At the top of the left sidebar, together with the app title.
2. In the **AI laboratory** tab under **Project team**.

## Main features

- Natural-language preference parsing
- Weighted and explainable city ranking
- TF-IDF and cosine-similarity matching
- K-means city archetypes
- Three-hidden-layer MLP neural-network demonstration
- Like/dislike feedback with a Beta-Bernoulli bandit-style adjustment
- Interactive Folium/OpenStreetMap visualization
- Plotly comparisons and radar charts
- Optional live weather and Wikipedia context

## Run locally

```bash
python -m venv .venv
```

Activate the environment:

```bash
# Windows PowerShell
.venv\Scripts\Activate.ps1

# macOS or Linux
source .venv/bin/activate
```

Install and run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

## Deploy with Streamlit Community Cloud

1. Extract this ZIP file.
2. Create a new GitHub repository.
3. Upload all files and folders from the extracted repository into the repository root.
4. In Streamlit Community Cloud, select the GitHub repository.
5. Set the main file path to `app.py`.
6. Deploy the app.

The following files must stay together in the repository root:

```text
akbarxon-ai-living-advisor-updated/
├── .streamlit/
│   └── config.toml
├── .gitignore
├── app.py
├── cities_uzbekistan.csv
├── requirements.txt
├── README.md
└── Akbarxon_AI_Living_Advisor_Uzbekistan_Updated_Colab.ipynb
```

## Data and model disclaimer

The included city scores are illustrative prototype values rather than official measurements. Before a public release, replace them with licensed, dated, auditable data and show the source, retrieval date, geographic scope, and confidence for each metric.

This educational prototype does not provide legal, immigration, housing, medical, employment, or financial advice.


In [ ]:
%%writefile .gitignore
__pycache__/
*.py[cod]
.venv/
.env
.streamlit/secrets.toml
feedback_state.json
demo_preference_dnn.joblib
.DS_Store


In [ ]:
from pathlib import Path

Path('.streamlit').mkdir(exist_ok=True)
Path('.streamlit/config.toml').write_text(
    '''[theme]
base = "light"
primaryColor = "#ff4b5c"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f3f6fb"
textColor = "#172033"
font = "sans serif"

[server]
headless = true

[browser]
gatherUsageStats = false
''',
    encoding='utf-8',
)
print('Wrote .streamlit/config.toml')


## 5. Validate the generated project

This cell checks Python syntax, verifies the required files, trains the demonstration neural network, and runs two recommendation examples.

In [ ]:
import py_compile
from pathlib import Path

required_files = [
    Path('app.py'),
    Path('cities_uzbekistan.csv'),
    Path('requirements.txt'),
    Path('README.md'),
    Path('.gitignore'),
]

for file_path in required_files:
    assert file_path.exists(), f'Missing required file: {file_path}'

py_compile.compile('app.py', doraise=True)
print('✅ app.py syntax validation passed.')
print('✅ All required project files exist.')

In [ ]:
import app

city_df = app.load_city_data()
feedback = app.load_feedback_state(city_df['city'])
zero_sliders = {feature: 0.0 for feature in app.FEATURE_COLUMNS}

example_queries = [
    'cheapest living city with the cheapest grocery price',
    'I want a city with perfect sunset views and great entertainment activities',
]

for query in example_queries:
    ranked, weights, matches = app.rank_cities(
        city_df,
        query=query,
        slider_weights=zero_sliders,
        feedback_state=feedback,
        top_n=5,
    )
    print('
QUERY:', query)
    display(ranked[['city', 'region', 'match_score', 'preference_score', 'dnn_like_probability']])
    print('Detected intent:', matches)

print('✅ Recommendation smoke tests completed.')

## 6. Package everything as a GitHub-uploadable ZIP

The notebook remains the single source that creates the project. This cell packages the generated website files for convenience.

In [ ]:
import shutil
from pathlib import Path

repo_dir = Path('akbarxon-ai-living-advisor-updated')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(exist_ok=True)
(repo_dir / '.streamlit').mkdir(exist_ok=True)

for filename in [
    'app.py',
    'cities_uzbekistan.csv',
    'requirements.txt',
    'README.md',
    '.gitignore',
    'Akbarxon_AI_Living_Advisor_Uzbekistan_Updated_Colab.ipynb',
]:
    source = Path(filename)
    if source.exists():
        shutil.copy2(source, repo_dir / filename)

shutil.copy2('.streamlit/config.toml', repo_dir / '.streamlit' / 'config.toml')

zip_path = shutil.make_archive(
    base_name='akbarxon-ai-living-advisor-updated',
    format='zip',
    root_dir=repo_dir,
)
print('✅ Created:', zip_path)


In [ ]:
# In Google Colab, run this cell to download the generated GitHub package.
try:
    from google.colab import files
    files.download('akbarxon_ai_living_advisor_github.zip')
except ImportError:
    print('Not running in Colab. The ZIP file is available in the current working directory.')

## 7. Preview the Streamlit website inside Colab

Run the next cell. Colab will display a clickable proxy URL. Keep the cell's Streamlit process running while testing the app.

In [ ]:
import os
import subprocess
import time

# Stop an older preview process if this cell is rerun.
subprocess.run(['pkill', '-f', 'streamlit run app.py'], check=False)

log_file = open('streamlit.log', 'w')
process = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--browser.gatherUsageStats', 'false',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(4)

try:
    from google.colab import output
    preview_url = output.eval_js('google.colab.kernel.proxyPort(8501)')
    print('Open the Streamlit preview:', preview_url)
except ImportError:
    print('Streamlit is running at http://localhost:8501')

print('Process ID:', process.pid)

In [ ]:
# Optional: inspect the Streamlit startup log if the preview does not open.
print(Path('streamlit.log').read_text(encoding='utf-8')[-4000:])

## 8. Upload to GitHub and deploy on Streamlit Community Cloud

1. Create a new public GitHub repository, for example `akbarxon-ai-living-advisor`.
2. Extract `akbarxon_ai_living_advisor_github.zip`.
3. Upload the extracted files to the repository root.
4. Open Streamlit Community Cloud and create a new app from that repository.
5. Select `app.py` as the entry-point file.
6. Deploy and test the map, recommendation examples, live context, and feedback controls.

### Files that must remain together

```text
akbarxon-ai-living-advisor/
├── app.py
├── cities_uzbekistan.csv
├── requirements.txt
├── README.md
└── .gitignore
```

### Recommended production upgrades

- Replace prototype city scores with licensed city-level housing, food, transport, healthcare, education, safety, air-quality, weather, and employment data.
- Store the source URL, source organization, retrieval date, geographic scope, and confidence for every metric.
- Replace local feedback JSON with PostgreSQL, Supabase, or Firebase.
- Add user accounts, consent, deletion requests, rate limiting, moderation, and audit logs.
- Add Uzbek and Russian translations.
- Add a retrieval pipeline with a search API, source quality ranking, deduplication, citation generation, and stale-data detection.
- Evaluate recommendation accuracy, subgroup fairness, privacy, and harmful relocation advice before public release.

### Ethical and legal guardrails

The platform should never present prototype scores as facts or guarantee that a location is safe, cheap, medically appropriate, or suitable for immigration. Users should receive source links, dates, uncertainty indicators, and reminders to verify housing, employment, visa, healthcare, and legal information independently.

## 9. Technical references

- Google Colab runtime FAQ: current runtime images and Python versions
- Streamlit release notes and deployment documentation
- Scikit-learn documentation for TF–IDF, cosine similarity, K-means, and `MLPClassifier`
- Folium documentation for interactive Leaflet/OpenStreetMap maps
- Open-Meteo API documentation for public weather context
- MediaWiki REST API documentation for city summaries

The application deliberately uses an explainable hybrid architecture. The weighted score remains dominant, so users can understand why a city was recommended rather than receiving an unexplained black-box answer.